# Stage 2 Detection — Inference Benchmark

Runs `model.val()` on the original `autosplit_val.txt` split for each stage-2 model (n/s/m/l/x) and plots **mAP@50 vs inference time**.

> Val set is `autosplit_val.txt` — the same split used during training — to avoid data leakage.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

from pathlib import Path

RUNS_DIR    = Path("runs/detect/stage2_detect_target")   # relative to notebook
SIZES       = ["n", "s", "m", "l", "x"]

# yaml must point to autosplit_val.txt for the val split
EVAL_YAML   = "2025_pollen.yaml"

IMGSZ       = 640
BATCH       = 16
DEVICE      = "0"
SINGLE_CLS  = True    # must match training config

PLOTS_DIR   = Path("../plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from ultralytics import YOLO

In [ ]:
# ── Run val for each model ────────────────────────────────────────────────────
records = []

for size in SIZES:
    weights = RUNS_DIR / f"yolo26{size}_detect_target" / "weights" / "best.pt"
    assert weights.exists(), f"Weights not found: {weights}"

    print(f"\n{'='*55}")
    print(f" Evaluating  yolo26{size}  ({weights})")
    print(f"{'='*55}")

    model   = YOLO(str(weights))
    results = model.val(
        data       = EVAL_YAML,
        imgsz      = IMGSZ,
        batch      = BATCH,
        device     = DEVICE,
        single_cls = SINGLE_CLS,
        verbose    = True,
    )

    map50          = float(results.box.map50)
    infer_ms       = float(results.speed["inference"])   # ms / image
    preproc_ms     = float(results.speed["preprocess"])
    postproc_ms    = float(results.speed["postprocess"])

    rec = {
        "model"         : f"yolo26{size}",
        "size"          : size,
        "mAP50"         : round(map50, 4),
        "infer_ms"      : round(infer_ms, 3),
        "preproc_ms"    : round(preproc_ms, 3),
        "postproc_ms"   : round(postproc_ms, 3),
        "total_ms"      : round(preproc_ms + infer_ms + postproc_ms, 3),
    }
    records.append(rec)
    print(f"  mAP@50={map50:.4f}   infer={infer_ms:.2f} ms/img")

df = pd.DataFrame(records).set_index("model")
print("\nSummary:")
print(df[["mAP50", "infer_ms", "total_ms"]].to_string())

In [ ]:
# ── Save results to CSV ───────────────────────────────────────────────────────
csv_path = PLOTS_DIR / "stage2_inference_benchmark.csv"
df.to_csv(csv_path)
print(f"Saved → {csv_path}")

In [ ]:
# ── Plot: mAP@50 vs inference time ────────────────────────────────────────────
SIZE_ORDER = {"n": 0, "s": 1, "m": 2, "l": 3, "x": 4}
COLORS     = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]

fig, ax = plt.subplots(figsize=(7, 5))

for i, row in df.reset_index().iterrows():
    ax.scatter(
        row["infer_ms"], row["mAP50"],
        s=120, color=COLORS[i], zorder=3, label=row["model"],
    )
    ax.annotate(
        row["model"],
        xy=(row["infer_ms"], row["mAP50"]),
        xytext=(6, 4), textcoords="offset points",
        fontsize=9,
        path_effects=[pe.withStroke(linewidth=2, foreground="white")],
    )

# Connect points in size order (n → s → m → l → x)
df_sorted = df.reset_index().sort_values("infer_ms")
ax.plot(df_sorted["infer_ms"], df_sorted["mAP50"],
        color="gray", linewidth=1, linestyle="--", zorder=2, alpha=0.6)

ax.set_xlabel("Inference time (ms / image)", fontsize=11)
ax.set_ylabel("mAP@50", fontsize=11)
ax.set_title("Stage 2 Detection — mAP@50 vs Inference Time\n"
             "(autosplit_val, single_cls=True, imgsz=640)", fontsize=11)
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
plot_path = PLOTS_DIR / "stage2_map50_vs_infer_time.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved → {plot_path}")